In [1]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import json
import torch
import hashlib
print(torch.cuda.is_available())
from datasets import load_dataset
from rapidfuzz import fuzz
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForTokenClassification,pipeline

True


In [ ]:
pip install rapidfuzz

In [2]:
dataset = load_dataset("SetFit/enron_spam")
print(pd.DataFrame(dataset["train"]).head())

Repo card metadata block was not found. Setting CardData to empty.


   message_id                                               text  label  \
0       33214  any software just for 15 $ - 99 $ understandin...      1   
1       11929  perspective on ferc regulatory action client c...      0   
2       19784  wanted to try ci 4 lis but thought it was way ...      1   
3        2209  enron / hpl actuals for december 11 , 2000 tec...      0   
4       15880  looking for cheap high - quality software ? ro...      1   

  label_text                                            subject  \
0       spam                  any software just for 15 $ - 99 $   
1        ham  perspective on ferc regulatory action client c...   
2       spam  wanted to try ci 4 lis but thought it was way ...   
3        ham         enron / hpl actuals for december 11 , 2000   
4       spam  looking for cheap high - quality software ? ro...   

                                             message       date  
0  understanding oem software\nlead me not into t... 2005-06-18  
1  19 th , 2 :

In [3]:
#Data preprocessing with artifact dedup
def preprocess_function(row):

    subject = row.get("subject", "") or ""
    message = row.get("message", "") or ""

    raw_text = subject + "\n" + message
    text = (subject + "\n" + message).strip()
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)
    # Remove quoted email chains
    text = re.sub(r"On .* wrote:", "", text)
    text = re.sub(r">.*", "", text)
    #lowercase normalization
    text = text.lower()
    
    timestamp = pd.to_datetime(row["date"], errors="coerce")

    # Artifact hash for deduplication
    artifact_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()

    # Thread ID
    thread_id = row.get("thread_id", row.get("message_id"))

    # Filter very short artifacts
    if len(text) < 20:
        return None

    return {
        "source_id": row.get("message_id"),
        "thread_id": thread_id,
        "artifact_hash": artifact_hash,
        "timestamp": timestamp,
        "text": text,
        "raw_text": raw_text
    }


dataset = dataset.map(preprocess_function)
dataset_initial = dataset.remove_columns(['message_id','label','label_text','subject','message','date'])
print(pd.DataFrame(dataset_initial["train"]).head())

                                                text  source_id  thread_id  \
0  any software just for 15 $ - 99 $ understandin...      33214      33214   
1  perspective on ferc regulatory action client c...      11929      11929   
2  wanted to try ci 4 lis but thought it was way ...      19784      19784   
3  enron / hpl actuals for december 11 , 2000 tec...       2209       2209   
4  looking for cheap high - quality software ? ro...      15880      15880   

                                       artifact_hash  timestamp  \
0  f26e101a9f3e6cfdbb6598f338ff9a37fe097c66381a46... 2005-06-18   
1  8298e58b82116ab575d6541a4f918dc4490b845c19afb9... 2001-06-19   
2  88be84ee6db3b75a0dbc9b49a042bf12621aacc652f2b5... 2004-09-11   
3  6d6a6399419f810fd185cfffe1a356324b9f7313c55586... 2000-12-12   
4  db44c8c964d6d58271fe8374244c4e905238d5410e8302... 2005-02-13   

                                            raw_text  
0  any software just for 15 $ - 99 $\nunderstandi...  
1  perspective on 

In [4]:
# Entity extraction NER-BERT models
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
ner_pipeline = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)
def extract_entities_batch(batch):
    ner_results = ner_pipeline(batch["text"])   # processes a list of texts

    all_entities = []

    for results in ner_results:
        entities = []
        for result in results:
            entity = {
                "entity": result.get("entity_group"),
                "score": result.get("score"),
                "start": result.get("start"),
                "end": result.get("end"),
                "word": result.get("word")
            }
            entities.append(entity)

        all_entities.append(entities)

    return {"entities": all_entities}

dataset_initial = dataset_initial.map(
    extract_entities_batch,
    batched=True,
    batch_size=16
)
print(dataset_initial.column_names)
dataset_initial = dataset_initial.filter(
    lambda x: len(x["entities"]) > 0
)
print(pd.DataFrame(dataset_initial["train"]).head())
    

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1993 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'train': ['text', 'source_id', 'thread_id', 'artifact_hash', 'timestamp', 'raw_text', 'entities'], 'test': ['text', 'source_id', 'thread_id', 'artifact_hash', 'timestamp', 'raw_text', 'entities']}


Filter:   0%|          | 0/1993 [00:00<?, ? examples/s]

                                                text  source_id  thread_id  \
0  emerging growth stock profile vera , vcsc - br...      15726      15726   
1  fortune most admired ranking congratulations !...       5458       5458   
2  re : risk position - eugenio perez ? ? ? ? tha...      22721      22721   
3  delainey is everywhere . he is working on doug...      14691      14691   
4  hpl books and records sally and brian , per my...      24028      24028   

                                       artifact_hash  timestamp  \
0  ccd2fbd9605c40ff936bcdbf443e8cc253aa7170fb3f3a... 2005-01-18   
1  135f21549a330556c5a32adfb57cb0c0502cf39be579fe... 2000-02-07   
2  4de69d9c13d668a7cc43c03d433937098a6d914c3a87eb... 2000-03-29   
3  6511aace8f5a7e6f6ec1fd5a169049a6a8fbbad13a1e8e... 2002-01-10   
4  b6291e05ff3ca7ab3f63bb703f135ce6e8d4b2c78f0063... 2001-04-19   

                                            raw_text  \
0  emerging growth stock profile\nvera ,\nvcsc - ...   
1  fortune most 

In [5]:
# Functions for entity canonicalization
def normalize_name(name):

    name = name.lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name
entity_registry = {}
entity_mentions = []

entity_counter = defaultdict(int)

# optional known aliases
alias_dict = {
    "ml team": "machine learning team",
    "ai team": "artificial intelligence team"
}
def find_existing_entity(name, label, threshold=90):

    for entity_id, entity in entity_registry.items():

        if entity["type"] != label:
            continue

        # check canonical name
        score = fuzz.ratio(name, entity["canonical_name"])

        if score >= threshold:
            return entity_id

        # check aliases
        for alias in entity["aliases"]:
            if fuzz.ratio(name, normalize_name(alias)) >= threshold:
                return entity_id

    return None
def create_new_entity(name, label):

    entity_counter[label] += 1
    entity_id = f"{label.lower()}_{entity_counter[label]}"

    entity_registry[entity_id] = {
        "entity_id": entity_id,
        "canonical_name": name,
        "type": label,
        "aliases": set()
    }

    return entity_id
def canonicalize_entities(row):

    entities = row["entities"]
    source_id = row["source_id"]

    canonical_entities = []
    seen = set()

    for e in entities:

        raw_name = e["word"]
        label = e["entity"]

        normalized = normalize_name(raw_name)

        # resolve known alias
        if normalized in alias_dict:
            normalized = alias_dict[normalized]

        key = (normalized, label)

        if key in seen:
            continue

        seen.add(key)

        entity_id = find_existing_entity(normalized, label)

        if entity_id is None:
            entity_id = create_new_entity(normalized, label)

        # automatically store alias
        entity_registry[entity_id]["aliases"].add(raw_name)

        entity_mentions.append({
            "entity_id": entity_id,
            "mention": raw_name,
            "normalized": normalized,
            "type": label,
            "source_id": source_id
        })

        canonical_entities.append({
            "entity_id": entity_id,
            "canonical_name": entity_registry[entity_id]["canonical_name"],
            "type": label
        })

    return {"canonical_entities": canonical_entities}
dataset_initial = dataset_initial.map(canonicalize_entities)
entities_df = pd.DataFrame(entity_registry.values())
mentions_df = pd.DataFrame(entity_mentions)
print(pd.DataFrame(dataset_initial["train"][:3]))
print(entities_df.head())
print(mentions_df.head())

Map:   0%|          | 0/222 [00:00<?, ? examples/s]

                                                text  source_id  thread_id  \
0  emerging growth stock profile vera , vcsc - br...      15726      15726   
1  fortune most admired ranking congratulations !...       5458       5458   
2  re : risk position - eugenio perez ? ? ? ? tha...      22721      22721   

                                       artifact_hash  timestamp  \
0  ccd2fbd9605c40ff936bcdbf443e8cc253aa7170fb3f3a... 2005-01-18   
1  135f21549a330556c5a32adfb57cb0c0502cf39be579fe... 2000-02-07   
2  4de69d9c13d668a7cc43c03d433937098a6d914c3a87eb... 2000-03-29   

                                            raw_text  \
0  emerging growth stock profile\nvera ,\nvcsc - ...   
1  fortune most admired ranking\ncongratulations ...   
2  re : risk position - eugenio perez\n? ? ? ? th...   

                                            entities  \
0  [{'end': 934, 'entity': 'ORG', 'score': 0.8844...   
1  [{'end': 92, 'entity': 'ORG', 'score': 0.75015...   
2  [{'end': 556, 'entity'

# Now relation extraction

In [ ]:
pip install --upgrade transformers

In [29]:
import json
import re
import uuid
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

model_name = "t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

PROMPT_VERSION = "claim_extraction_v1"


# ------------------------------------------------
# Prompt
# ------------------------------------------------
def build_prompt(email_text, entities):

    entity_names = [e["canonical_name"] for e in entities]

    return f"""
Extract factual relations from the text.

Text:
{email_text}

Entities:
{entity_names}

Return ONLY valid JSON list with objects having:
subject, relation, object, evidence_text, evidence_offsets

Example:
[
 {{
  "subject": "Tim Cook",
  "relation": "works_at",
  "object": "Apple",
  "evidence_text": "Tim Cook joined Apple in 1998",
  "evidence_offsets": [10,35]
 }}
]

If no relations exist return []

JSON:
"""


# ------------------------------------------------
# Safe JSON parser
# ------------------------------------------------
def safe_json_parse(text):

    text = text.strip()

    try:
        return json.loads(text)
    except:
        pass

    # Try extracting JSON list
    matches = re.findall(r"\[[\s\S]*?\]", text)

    for m in matches:
        try:
            return json.loads(m)
        except:
            continue

    return []


# ------------------------------------------------
# Claim extraction
# ------------------------------------------------
def extract_relations_batch(email_texts, email_entities, source_ids, max_tokens=200):

    batch_results = []

    for text, entities, source_id in zip(email_texts, email_entities, source_ids):

        if not entities:
            entities = [{"canonical_name": "document"}]

        prompt = build_prompt(text, entities)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False
        )

        out = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Debug
        print("\nMODEL OUTPUT:\n", out)

        claims = safe_json_parse(out)

        parsed_claims = []

        for c in claims:

            if not isinstance(c, dict):
                continue

            subject = c.get("subject")
            relation = c.get("relation")
            obj = c.get("object")

            if subject and relation and obj:

                parsed_claims.append({
                    "claim_id": str(uuid.uuid4()),
                    "subject": subject,
                    "relation": relation,
                    "object": obj,
                    "evidence_text": c.get("evidence_text"),
                    "evidence_offsets": c.get("evidence_offsets"),
                    "source_id": source_id,
                    "prompt_version": PROMPT_VERSION
                })

        batch_results.append(parsed_claims)

    return batch_results


# ------------------------------------------------
# Run extraction over dataset
# ------------------------------------------------
batch_size = 16
MAX_STEPS = 70

splits = list(dataset_initial.keys())

for split in splits:

    dataset_split = dataset_initial[split]

    claims_list = []

    for step, start_idx in enumerate(
        tqdm(range(0, len(dataset_split), batch_size),
        desc=f"Claim extraction ({split})")
    ):

        if step >= MAX_STEPS:
            break

        batch = dataset_split.select(
            range(start_idx, min(start_idx + batch_size, len(dataset_split)))
        )

        MAX_CHARS = 1500

        email_texts = [row["text"][:MAX_CHARS] for row in batch]
        email_entities = [row.get("canonical_entities", []) for row in batch]
        source_ids = [row.get("source_id") for row in batch]

        batch_claims = extract_relations_batch(
            email_texts,
            email_entities,
            source_ids
        )

        for claims in batch_claims:
            claims_list.append(claims)

    # Keep only rows we processed
    dataset_small = dataset_split.select(range(len(claims_list)))

    # Remove old claims column if it exists
    if "claims" in dataset_small.column_names:
        dataset_small = dataset_small.remove_columns("claims")

    # Add new claims column
    dataset_small = dataset_small.add_column("claims", claims_list)

    dataset_initial[split] = dataset_small

print("Claim extraction completed for limited batches.")

# ------------------------------------------------
# Preview
# ------------------------------------------------
print(dataset_initial["train"].to_pandas().head())

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Claim extraction (train):   0%|          | 0/223 [00:00<?, ?it/s]


MODEL OUTPUT:
 vocalscape networks inc . is building a company that ' s revoiutionizing the teiecommunications industry with the most affordable phone systems . vocaiscape has developed software and interactive soiutions revoiutionizing the giobal communications industry .

MODEL OUTPUT:
 fortune magazine has named enron the " most innovative company in america " for the fifth year in a row . enron has also been ranked # 1 in " quality of management , topping general electric and omnicom group . enron's " employee talent " has been ranked # 2 , behind goldman sachs and ahead of cisco systems .

MODEL OUTPUT:
 i am in london this week and have picked up your message with the attached . in light of your response i think i ' m after 2 things : 1 . a spec that outlines eugenio ' s role . and 2 . a spec that could be used to hire a japanese national to replace eugenio at the end of eugenio ' s assignment in japan .

MODEL OUTPUT:
 True

MODEL OUTPUT:
 hpl / aep records transfer list will b

Claim extraction (train):   0%|          | 1/223 [00:20<1:14:59, 20.27s/it]


MODEL OUTPUT:
 cm - buffett winter tour dates january 2001 february 2001 . jimmy buffett winter tour 2001 february . all dates are official , if any more are released we ' ll let you know .

MODEL OUTPUT:
 jim khan: i have been disturbed by our lack of information / data relative to our position in the grid and the overall fundamentals for the region . he will be gone for 3 weeks starting wednesday . khan: i would like to meet with you and get your assistance in identifying some resources to try and help the india team develop a better understanding of their market .

MODEL OUTPUT:
 kevin garland has been with dell for 6 years and is interested in joining enron ( specifically the trading / origination side of enron . he might fit best in ena - louise kitchen or dave duran . kevin garland - - - - - - - michael _ green @ dell . com 03 / 16 / 01 01 01 01 01 01 01 01 01 01 01 01 01 01

MODEL OUTPUT:
 olivus is a delicious tea that fights colds , flus , and other viruses and bacteria . oli

...


MODEL OUTPUT:
 epgt will be processing the s . g marshall gas april 2001 . therefore eogr will have to reduce the volume that will be available for sale to hpl . the new estimate is 3 , 400 mmbtu / day .

MODEL OUTPUT:
 bob hall is leaving netco and i think we need to identify and communicate who the lead is so that the team can be given direction . sally beck ' s world doesn ' t seem to have much direction . i think with bob ' s departure , we need to identify and communicate who the lead is so that the team can get things back on track .

MODEL OUTPUT:
 if no relations exist return ONLY valid JSON list with objects having: subject, relation, object, evidence_text, evidence_offsets . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist return . if no relations exist retu

Claim extraction (test):  93%|█████████▎| 13/14 [04:33<00:21, 21.06s/it]


MODEL OUTPUT:
 ene's quick read lq : 01 vs . $ 0 . 40 last year . wholesale volume up 65 % . ene's lq: 76 % . ene's lq: $ 1 . 75 - $ 1 . 80 . ene's lq: 0 . 47 vs $ 0 . 40 last year .

MODEL OUTPUT:
 e - bookservices is a professional outsourcing partner for all book and journal - related work . located in india , we cater to both organizations as well as individuals . our services include , but are not limited to : book writing and typesetting services .

MODEL OUTPUT:
 'ciallis' is better than pfizer viiagrra and normal ci - ialis softabs because : - guaaraantees 36 hours lasting - safe to take , no side effects at all - boost and increase se - xual performance - haarder e - rectiions and quick recharge - proven and certified by experts and doctors - only $ 3 . 99 per tabs cllick here

MODEL OUTPUT:
 a complimentary copy of the western us natural gas market review is attached . if no relations exist return ONLY valid JSON list with objects having: subject, relation, object, evidence_

Claim extraction (test): 100%|██████████| 14/14 [04:53<00:00, 20.96s/it]


MODEL OUTPUT:
 john rohde purchases gas from 30 small to medium sized suppliers in the rocky mountain area and either acts as their agent or moves their purchased gas to markets for ngts . john also indicated that he thought ngts does have some equity san juan gas which currently flows on el paso to california via capacity release . john also indicated that he thought ngts does have some equity san juan gas which currently flows
Claim extraction completed for limited batches.
                                                text  source_id  thread_id  \
0  emerging growth stock profile vera , vcsc - br...      15726      15726   
1  fortune most admired ranking congratulations !...       5458       5458   
2  re : risk position - eugenio perez ? ? ? ? tha...      22721      22721   
3  delainey is everywhere . he is working on doug...      14691      14691   
4  hpl books and records sally and brian , per my...      24028      24028   

                                       artifact_h

In [ ]:
print(dataset_initial["train"].to_pandas()[40:60])

In [30]:
import json
from collections import defaultdict
import uuid


# --------------------------------------------------
# Load claims from dataset
# --------------------------------------------------
def load_claims(dataset_split):

    all_claims = []

    for row in dataset_split:

        claims = row.get("claims", [])

        # If claims stored as JSON string
        if isinstance(claims, str):
            try:
                claims = json.loads(claims)
            except:
                claims = []

        # Ensure list
        if isinstance(claims, list):
            all_claims.extend(claims)

    return all_claims


# --------------------------------------------------
# Entity linking
# --------------------------------------------------
def link_entities(claims, entity_registry):

    linked_claims = []

    for c in claims:

        subject = c.get("subject")
        obj = c.get("object")

        c["subject_id"] = entity_registry.get(str(subject).lower())
        c["object_id"] = entity_registry.get(str(obj).lower())

        linked_claims.append(c)

    return linked_claims


# --------------------------------------------------
# Relation normalization
# --------------------------------------------------
def normalize_relations(claims, relation_map):

    for c in claims:

        rel = c.get("relation")

        if rel is not None:
            c["relation"] = relation_map.get(rel.lower(), rel)

    return claims


# --------------------------------------------------
# Deduplication
# --------------------------------------------------
merge_log = []

def deduplicate_claims(claims):

    merged = {}

    for c in claims:

        # Skip invalid claims
        if not c.get("subject_id") or not c.get("object_id") or not c.get("relation"):
            continue

        key = (c.get("subject_id"), c.get("relation"), c.get("object_id"))

        if key not in merged:

            merged[key] = c.copy()

            merged[key]["evidence_texts"] = [c.get("evidence_text")]

        else:

            merged[key]["evidence_texts"].append(c.get("evidence_text"))

            merge_log.append({
                "original_claim_id": c.get("claim_id"),
                "merged_into": merged[key].get("claim_id"),
                "reason": "deduplication"
            })

    return list(merged.values())


# --------------------------------------------------
# Conflict detection
# --------------------------------------------------
def detect_conflicts(claims):

    relation_map = defaultdict(list)

    for c in claims:

        key = (c.get("subject_id"), c.get("relation"))

        relation_map[key].append(c)

    conflicts = []

    for key, claim_list in relation_map.items():

        objects = {c.get("object_id") for c in claim_list}

        if len(objects) > 1:

            conflicts.append({
                "subject_id": key[0],
                "relation": key[1],
                "objects": list(objects),
                "claims": claim_list
            })

    return conflicts


# --------------------------------------------------
# Load claims from dataset splits
# --------------------------------------------------
all_claims = []

for split in splits:

    claims = load_claims(dataset_initial[split])

    all_claims.extend(claims)


# --------------------------------------------------
# Example entity registry
# --------------------------------------------------
entity_registry = {
    "apple": 1,
    "tim cook": 2,
    "steve jobs": 3
}


# --------------------------------------------------
# Relation normalization map
# --------------------------------------------------
relation_map = {
    "is_ceo_of": "CEO_of",
    "chief_executive_of": "CEO_of",
    "ceo_of": "CEO_of",
    "works_at": "employee_of"
}


# --------------------------------------------------
# Pipeline execution
# --------------------------------------------------
linked_claims = link_entities(all_claims, entity_registry)

normalized_claims = normalize_relations(linked_claims, relation_map)

unique_claims = deduplicate_claims(normalized_claims)

conflicts = detect_conflicts(unique_claims)


# --------------------------------------------------
# Metrics and debugging output
# --------------------------------------------------
print("Total claims loaded:", len(all_claims))

print("Total unique claims after deduplication:", len(unique_claims))

print("Total conflicts detected:", len(conflicts))


print("\nExample merged claim:")

if unique_claims:

    example = unique_claims[0]

    print("Subject ID:", example.get("subject_id"))
    print("Relation:", example.get("relation"))
    print("Object ID:", example.get("object_id"))
    print("Evidence texts:", example.get("evidence_texts"))


print("\nMerge log entries:", len(merge_log))

if merge_log:

    print("Example merge log entry:", merge_log[0])


print("\nExample conflict entry:", conflicts[0] if conflicts else "No conflicts")

Total claims loaded: 0
Total unique claims after deduplication: 0
Total conflicts detected: 0

Example merged claim:

Merge log entries: 0

Example conflict entry: No conflicts


In [ ]:
pip install pyvis networkx neo4j datetime

In [22]:
import re
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML

# -----------------------------
# Initialize graph
# -----------------------------
G = nx.DiGraph()

processed = 0
skipped = 0


# -----------------------------
# Helper to create node ids
# -----------------------------
def make_id(text):

    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9]+', '_', text)

    return text


# -----------------------------
# Extract fields safely
# -----------------------------
def extract_claim_fields(claim):

    subject = (
        claim.get("subject")
        or claim.get("subject_name")
        or claim.get("subj")
    )

    obj = (
        claim.get("object")
        or claim.get("object_name")
        or claim.get("obj")
    )

    relation = (
        claim.get("relation")
        or claim.get("predicate")
        or claim.get("rel")
    )

    evidence = claim.get("evidence_texts") or claim.get("evidence") or []

    return subject, relation, obj, evidence


# -----------------------------
# Add claim to graph
# -----------------------------
def add_claim(claim):

    global processed, skipped

    subject, relation, obj, evidence = extract_claim_fields(claim)

    if subject is None or obj is None or relation is None:
        skipped += 1
        return

    subject_id = make_id(subject)
    object_id = make_id(obj)

    G.add_node(subject_id, label=subject)
    G.add_node(object_id, label=obj)

    # Prevent evidence overwrite
    if G.has_edge(subject_id, object_id):

        existing = G[subject_id][object_id]["evidence"]
        G[subject_id][object_id]["evidence"] = list(set(existing + evidence))

    else:

        G.add_edge(
            subject_id,
            object_id,
            relation=relation,
            evidence=evidence
        )

    processed += 1


# -----------------------------
# Load all claims into graph
# -----------------------------
def load_graph_from_claims(all_claims):

    for claim in all_claims:
        add_claim(claim)

    print("Claims processed:", processed)
    print("Claims skipped:", skipped)
    print("Total nodes:", G.number_of_nodes())
    print("Total edges:", G.number_of_edges())


# -----------------------------
# Retrieve relevant claims
# -----------------------------
def retrieve_claims(question):

    question = question.lower()
    tokens = set(question.split())

    results = []

    for node, data in G.nodes(data=True):

        name = data["label"].lower()
        name_tokens = set(name.split())

        if tokens.intersection(name_tokens):

            for neighbor in G.successors(node):

                edge = G[node][neighbor]

                results.append({
                    "subject": data["label"],
                    "relation": edge["relation"],
                    "object": G.nodes[neighbor]["label"],
                    "evidence": edge["evidence"]
                })

    return results


# -----------------------------
# Visualization
# -----------------------------
def visualize_claims(claims):

    net = Network(
        height="700px",
        width="100%",
        directed=True,
        notebook=True
    )

    # Better physics layout
    net.barnes_hut()

    for c in claims:

        net.add_node(c["subject"], label=c["subject"])
        net.add_node(c["object"], label=c["object"])

        evidence = "\n".join(c["evidence"]) if c["evidence"] else ""

        net.add_edge(
            c["subject"],
            c["object"],
            label=c["relation"],
            title=evidence
        )

    net.save_graph("graph.html")

    display(HTML("graph.html"))


# -----------------------------
# Query interface
# -----------------------------
def ask_question():

    while True:

        q = input("\nAsk a question (or type exit): ")

        if q.lower() == "exit":
            break

        claims = retrieve_claims(q)

        if not claims:
            print("No relevant claims found")
            continue

        print("\nRetrieved claims:")

        for c in claims[:10]:
            print(c["subject"], "--", c["relation"], "-->", c["object"])

        visualize_claims(claims)


# -----------------------------
# RUN GRAPH PIPELINE
# -----------------------------

print("Total claims available:", len(unique_claims))

load_graph_from_claims(unique_claims)

ask_question()

Total claims available: 3785
Claims processed: 0
Claims skipped: 3785
Total nodes: 0
Total edges: 0


KeyboardInterrupt: Interrupted by user